# Prediction Deep-Dive Analysis (XGBoost vs TFT)

This notebook compares raw probabilistic forecasts of **XGBoost** and **TFT** against ground truth for all relevant targets.

Targets covered:
- DA price
- aFRR capacity price (pos/neg)
- aFRR activation price (pos/neg)
- aFRR activation rate (pos/neg)

All plots use publication-quality settings for thesis reporting.


In [5]:
# Imports & plot style
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(context='paper', style='whitegrid', palette='colorblind')
plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'grid.alpha': 0.25,
})

QCOLS = [f'p{q:02d}' for q in range(10, 100, 10)]


In [6]:
# Configuration
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

MODEL_RUNS_ROOT = REPO_ROOT / 'artifacts/model_runs'
RUN_ID = None  # e.g. '2026-04-20T15-36-58Z'; keep None to auto-pick latest available
XGB_RUN_DIR_OVERRIDE = None  # optional explicit Path
TFT_RUN_DIR_OVERRIDE = None  # optional explicit Path

SPLIT = 'test'  # 'val' or 'test'
LEAD_TIME = 1

# Optional explicit truth path (if None, manifest + defaults are used)
TRUTH_PATH_OVERRIDE = None


def _resolve_run_dir(model_key: str, run_id=None, override=None) -> Path:
    if override is not None:
        p = Path(override)
        if p.exists():
            return p
        raise FileNotFoundError(f'Override run dir does not exist: {p}')

    if not MODEL_RUNS_ROOT.exists():
        raise FileNotFoundError(f'Model-runs root not found: {MODEL_RUNS_ROOT}')

    if run_id:
        candidates = [MODEL_RUNS_ROOT / run_id]
    else:
        candidates = sorted([p for p in MODEL_RUNS_ROOT.iterdir() if p.is_dir()], reverse=True)

    aliases = [model_key.lower()]
    if model_key.lower() == 'xgboost':
        aliases.append('xgb')

    generic_fallback = None
    for rd in candidates:
        pred_dir = rd / 'predictions'
        if not pred_dir.exists():
            continue

        # Prefer model-specific long predictions.
        for alias in aliases:
            if list(pred_dir.glob(f'{alias}*pred_*long*.parquet')):
                return rd

        # Keep latest generic run as fallback.
        if generic_fallback is None and list(pred_dir.glob('*pred_*long*.parquet')):
            generic_fallback = rd

    if generic_fallback is not None:
        print(f'[WARN] No model-specific predictions found for {model_key}; using generic run: {generic_fallback}')
        return generic_fallback

    raise FileNotFoundError(
        f'Could not resolve run directory with predictions for model={model_key}. '
        f'Searched under: {MODEL_RUNS_ROOT}'
    )


XGB_RUN_DIR = _resolve_run_dir('xgboost', run_id=RUN_ID, override=XGB_RUN_DIR_OVERRIDE)
TFT_RUN_DIR = _resolve_run_dir('tft', run_id=RUN_ID, override=TFT_RUN_DIR_OVERRIDE)

print(f'XGB_RUN_DIR: {XGB_RUN_DIR}')
print(f'TFT_RUN_DIR: {TFT_RUN_DIR}')

TARGET_SPECS = {
    'da_price': {
        'pred_col': 'pred_da_price',
        'bundle': 'da',
        'truth_candidates': ['target_da_price', 'da_price', 'target_da_price_h1'],
    },
    'afrr_capacity_price_pos': {
        'pred_col': 'pred_afrr_capacity_price_pos',
        'bundle': 'afrr',
        'truth_candidates': ['target_afrr_capacity_price_pos', 'afrr_capacity_price_pos', 'target_afrr_capacity_price_pos_h1'],
    },
    'afrr_capacity_price_neg': {
        'pred_col': 'pred_afrr_capacity_price_neg',
        'bundle': 'afrr',
        'truth_candidates': ['target_afrr_capacity_price_neg', 'afrr_capacity_price_neg', 'target_afrr_capacity_price_neg_h1'],
    },
    'afrr_activation_price_pos': {
        'pred_col': 'pred_afrr_activation_price_pos',
        'bundle': 'afrr',
        'truth_candidates': ['target_afrr_activation_price_vwap_pos', 'afrr_activation_price_vwap_pos', 'target_afrr_activation_price_vwap_pos_h1'],
    },
    'afrr_activation_price_neg': {
        'pred_col': 'pred_afrr_activation_price_neg',
        'bundle': 'afrr',
        'truth_candidates': ['target_afrr_activation_price_vwap_neg', 'afrr_activation_price_vwap_neg', 'target_afrr_activation_price_vwap_neg_h1'],
    },
    'afrr_activation_rate_pos': {
        'pred_col': 'pred_afrr_activation_rate_pos',
        'bundle': 'afrr',
        'truth_candidates': ['target_afrr_activation_rate_pos', 'afrr_activation_rate_pos', 'target_afrr_activation_rate_pos_h1', 'afrr_activation_rate', 'target_afrr_rate_h1'],
    },
    'afrr_activation_rate_neg': {
        'pred_col': 'pred_afrr_activation_rate_neg',
        'bundle': 'afrr',
        'truth_candidates': ['target_afrr_activation_rate_neg', 'afrr_activation_rate_neg', 'target_afrr_activation_rate_neg_h1', 'afrr_activation_rate', 'target_afrr_rate_h1'],
    },
}

TARGET_ORDER = list(TARGET_SPECS.keys())


[WARN] No model-specific predictions found for tft; using generic run: /Users/leori/Code/energyTrading/artifacts/model_runs/2026-04-20T15-36-58Z
XGB_RUN_DIR: /Users/leori/Code/energyTrading/artifacts/model_runs/2026-04-20T15-36-58Z
TFT_RUN_DIR: /Users/leori/Code/energyTrading/artifacts/model_runs/2026-04-20T15-36-58Z


## 1. Dynamic Data Loading & Alignment

In [7]:
@dataclass
class LoadedPred:
    run_dir: Path
    model_label: str
    pred_file: Path
    df: pd.DataFrame


def _load_manifest(run_dir: Path) -> dict[str, Any]:
    m = run_dir / 'manifest.json'
    if not m.exists():
        return {}
    return json.loads(m.read_text(encoding='utf-8'))


def _resolve_truth_path(run_dir: Path, manifest: dict[str, Any], override=None) -> Path:
    if override is not None:
        p = Path(override)
        if p.exists():
            return p
    mpath = manifest.get('ground_truth', {}).get('default_path', '')
    if mpath:
        p = Path(mpath)
        if p.exists():
            return p
    candidates = [
        REPO_ROOT / 'data/features/all_data_features.parquet',
        REPO_ROOT / 'data/model_input/afrr/test.parquet',
        REPO_ROOT / 'data/model_input/da/test.parquet',
    ]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError('Could not resolve a ground-truth parquet path.')


def _discover_prediction_file(
    run_dir: Path,
    manifest: dict[str, Any],
    pred_col: str,
    bundle: str,
    split: str,
    model_label: str,
) -> Path:
    pmap = (
        manifest.get('bundles', {})
        .get(bundle, {})
        .get('predictions_long', {})
        .get(split, {})
    )
    p = pmap.get(pred_col)
    if p:
        pp = Path(p)
        if pp.exists():
            return pp

    pred_dir = run_dir / 'predictions'
    if not pred_dir.exists():
        raise FileNotFoundError(f'Missing predictions directory: {pred_dir}')

    label = model_label.lower()
    aliases = [label]
    if label == 'xgb':
        aliases.append('xgboost')
    if label == 'xgboost':
        aliases.append('xgb')

    # Prefer model-specific long predictions first.
    patterns = []
    for alias in aliases:
        patterns.extend([
            f'{alias}*{split}*{pred_col}*long*.parquet',
            f'{alias}*{pred_col}*long*{split}*.parquet',
            f'{alias}*{pred_col}*long*.parquet',
        ])

    # Then fall back to generic long predictions.
    patterns.extend([
        f'*{split}*{pred_col}*long*.parquet',
        f'*{pred_col}*long*{split}*.parquet',
        f'*{pred_col}*long*.parquet',
    ])

    hits = []
    for pat in patterns:
        hits.extend(sorted(pred_dir.glob(pat)))

    uniq = []
    seen = set()
    for h in hits:
        if h in seen:
            continue
        seen.add(h)
        uniq.append(h)

    if not uniq:
        raise FileNotFoundError(
            f'No long prediction parquet found for model={model_label}, pred_col={pred_col}, split={split} in {pred_dir}'
        )

    return uniq[0]


def _load_prediction_long(run_dir: Path, model_label: str, pred_col: str, bundle: str, split: str, lead_time: int) -> LoadedPred:
    manifest = _load_manifest(run_dir)
    pred_file = _discover_prediction_file(
        run_dir,
        manifest,
        pred_col=pred_col,
        bundle=bundle,
        split=split,
        model_label=model_label,
    )
    df = pd.read_parquet(pred_file).copy()

    for c in ['snapshot_time_utc', 'target_time_utc']:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], utc=True, errors='coerce')

    if 'lead_time_h' in df.columns:
        df = df.loc[pd.to_numeric(df['lead_time_h'], errors='coerce') == int(lead_time)].copy()

    if 'target_time_utc' in df.columns:
        df['timestamp'] = df['target_time_utc']
    elif 'timestamp_utc' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp_utc'], utc=True, errors='coerce')
    else:
        raise KeyError(f'No time column found in prediction file: {pred_file.name}')

    if 'predicted_value' in df.columns and 'p50' not in df.columns:
        df['p50'] = pd.to_numeric(df['predicted_value'], errors='coerce')

    keep = ['timestamp', 'snapshot_time_utc', 'target_time_utc', 'lead_time_h', 'model_name'] + [c for c in QCOLS if c in df.columns] + [c for c in ['predicted_value', 'y_true'] if c in df.columns]
    df = df[sorted(set(keep), key=keep.index)].copy()

    prefix = model_label.lower()
    rename = {q: f'{prefix}_{q}' for q in QCOLS if q in df.columns}
    if 'predicted_value' in df.columns:
        rename['predicted_value'] = f'{prefix}_predicted_value'
    if 'y_true' in df.columns:
        rename['y_true'] = f'{prefix}_y_true'
    df = df.rename(columns=rename)

    return LoadedPred(run_dir=run_dir, model_label=model_label, pred_file=pred_file, df=df)


def _select_truth_column(truth_df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in truth_df.columns:
            return c
    raise KeyError(f'No truth column found. Tried: {candidates}')


def load_and_align_predictions(target_name: str, xgb_run_dir, tft_run_dir, split: str = 'test', lead_time: int = 1, truth_path_override=None) -> pd.DataFrame:
    # Load long predictions from XGB and TFT, align on target timestamp,
    # merge with ground truth, and return unified dataframe.
    if target_name not in TARGET_SPECS:
        raise KeyError(f'Unknown target_name={target_name}. Available: {list(TARGET_SPECS)}')

    spec = TARGET_SPECS[target_name]
    pred_col = spec['pred_col']
    bundle = spec['bundle']

    xgb_run_dir = Path(xgb_run_dir)
    tft_run_dir = Path(tft_run_dir)

    x = _load_prediction_long(xgb_run_dir, 'xgb', pred_col=pred_col, bundle=bundle, split=split, lead_time=lead_time)
    t = _load_prediction_long(tft_run_dir, 'tft', pred_col=pred_col, bundle=bundle, split=split, lead_time=lead_time)

    merged = pd.merge(x.df, t.df, on='timestamp', how='inner', suffixes=('_xgbmeta', '_tftmeta'))

    x_manifest = _load_manifest(xgb_run_dir)
    truth_path = _resolve_truth_path(xgb_run_dir, x_manifest, override=truth_path_override)
    truth_df = pd.read_parquet(truth_path).copy()

    if 'timestamp_utc' in truth_df.columns:
        truth_df['timestamp'] = pd.to_datetime(truth_df['timestamp_utc'], utc=True, errors='coerce')
    elif 'target_time_utc' in truth_df.columns:
        truth_df['timestamp'] = pd.to_datetime(truth_df['target_time_utc'], utc=True, errors='coerce')
    else:
        raise KeyError('Truth parquet must contain timestamp_utc or target_time_utc.')

    truth_col = _select_truth_column(truth_df, spec['truth_candidates'])
    truth_small = truth_df[['timestamp', truth_col]].copy().rename(columns={truth_col: 'y_true'})

    out = pd.merge(merged, truth_small, on='timestamp', how='left')

    if out['y_true'].isna().all():
        if 'xgb_y_true' in out.columns and not out['xgb_y_true'].isna().all():
            out['y_true'] = out['xgb_y_true']
        elif 'tft_y_true' in out.columns and not out['tft_y_true'].isna().all():
            out['y_true'] = out['tft_y_true']

    if 'xgb_p50' in out.columns:
        out['xgb_p50'] = pd.to_numeric(out['xgb_p50'], errors='coerce')
    if 'tft_p50' in out.columns:
        out['tft_p50'] = pd.to_numeric(out['tft_p50'], errors='coerce')
    out['y_true'] = pd.to_numeric(out['y_true'], errors='coerce')

    out = out.sort_values('timestamp').reset_index(drop=True)
    return out


In [8]:
# Quick sanity check for one target
df_check = load_and_align_predictions(
    target_name='da_price',
    xgb_run_dir=XGB_RUN_DIR,
    tft_run_dir=TFT_RUN_DIR,
    split=SPLIT,
    lead_time=LEAD_TIME,
    truth_path_override=TRUTH_PATH_OVERRIDE,
)

df_check[['timestamp', 'y_true', 'xgb_p50', 'tft_p50']].head()


,timestamp,y_true,xgb_p50,tft_p50
0,2024-07-04 01:00:00+00:00,10.08,34.553680,34.553680
1,2024-07-04 02:00:00+00:00,10.27,30.745483,30.745483
2,2024-07-04 03:00:00+00:00,36.43,28.883812,28.883812
3,2024-07-04 04:00:00+00:00,40.00,31.160355,31.160355
4,2024-07-04 05:00:00+00:00,33.56,71.429749,71.429749


## 2. Time-Series Excerpts (Micro-Patterns)
Ground truth is plotted as thick black line, with XGBoost and TFT medians plus P10–P90 bands.

In [9]:
def _highest_vol_window(df: pd.DataFrame, value_col: str = 'y_true', window_days: int = 7) -> tuple[pd.Timestamp, pd.Timestamp]:
    x = df[['timestamp', value_col]].dropna().sort_values('timestamp').copy()
    if x.empty:
        raise ValueError('No data available to determine volatility window.')
    x = x.set_index('timestamp')
    rolling_std = x[value_col].rolling(f'{window_days}D').std()
    end_ts = rolling_std.idxmax()
    if pd.isna(end_ts):
        end_ts = x.index.max()
    start_ts = end_ts - pd.Timedelta(days=window_days)
    return start_ts, end_ts


def plot_time_slice_with_bands(df: pd.DataFrame, target_name: str, start=None, end=None, window_days_if_auto: int = 7):
    d = df.copy()
    d = d.sort_values('timestamp')

    if start is None or end is None:
        start_ts, end_ts = _highest_vol_window(d, value_col='y_true', window_days=window_days_if_auto)
    else:
        start_ts = pd.to_datetime(start, utc=True)
        end_ts = pd.to_datetime(end, utc=True)

    w = d[(d['timestamp'] >= start_ts) & (d['timestamp'] <= end_ts)].copy()
    if w.empty:
        raise ValueError('Selected time window has no rows.')

    fig, ax = plt.subplots(figsize=(14, 5.5))

    if {'xgb_p10', 'xgb_p90'}.issubset(w.columns):
        ax.fill_between(w['timestamp'], w['xgb_p10'], w['xgb_p90'], alpha=0.18, color='tab:blue', label='XGB P10-P90')
    if {'tft_p10', 'tft_p90'}.issubset(w.columns):
        ax.fill_between(w['timestamp'], w['tft_p10'], w['tft_p90'], alpha=0.18, color='tab:orange', label='TFT P10-P90')

    ax.plot(w['timestamp'], w['y_true'], color='black', linewidth=2.4, label='Ground Truth')
    ax.plot(w['timestamp'], w['xgb_p50'], color='tab:blue', linestyle='--', linewidth=1.8, label='XGB P50')
    ax.plot(w['timestamp'], w['tft_p50'], color='tab:orange', linestyle=':', linewidth=2.0, label='TFT P50')

    ax.set_title(f'7-day Event Window (highest volatility) | {target_name}, lead={LEAD_TIME}')
    ax.set_xlabel('Timestamp (UTC)')
    ax.set_ylabel('Value')
    ax.legend(loc='best', ncol=2)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


plot_time_slice_with_bands(df_check, target_name='da_price')


/var/folders/mh/98qk5jbj4z336rv06wpgxbz40000gn/T/ipykernel_42034/598799236.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Residual & Bias Analysis

In [10]:
def plot_residual_histograms(df: pd.DataFrame, target_name: str):
    d = df[['y_true', 'xgb_p50', 'tft_p50']].dropna().copy()
    d['xgb_residual'] = d['xgb_p50'] - d['y_true']
    d['tft_residual'] = d['tft_p50'] - d['y_true']

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
    sns.histplot(d['xgb_residual'], bins=50, kde=True, color='tab:blue', ax=axes[0])
    axes[0].axvline(0, color='black', linewidth=1)
    axes[0].set_title(f'XGBoost Residuals | {target_name}')
    axes[0].set_xlabel('Prediction (P50) - True')

    sns.histplot(d['tft_residual'], bins=50, kde=True, color='tab:orange', ax=axes[1])
    axes[1].axvline(0, color='black', linewidth=1)
    axes[1].set_title(f'TFT Residuals | {target_name}')
    axes[1].set_xlabel('Prediction (P50) - True')

    plt.tight_layout()
    plt.show()


def plot_predicted_vs_actual_scatter(df: pd.DataFrame, target_name: str, sample_n: int = 12000, random_state: int = 42):
    d = df[['y_true', 'xgb_p50', 'tft_p50']].dropna().copy()
    if len(d) > sample_n:
        d = d.sample(sample_n, random_state=random_state)

    lo = np.nanmin([d['y_true'].min(), d['xgb_p50'].min(), d['tft_p50'].min()])
    hi = np.nanmax([d['y_true'].max(), d['xgb_p50'].max(), d['tft_p50'].max()])

    fig, ax = plt.subplots(figsize=(6.8, 6.5))
    ax.scatter(d['y_true'], d['xgb_p50'], s=12, alpha=0.35, label='XGB', color='tab:blue')
    ax.scatter(d['y_true'], d['tft_p50'], s=12, alpha=0.35, label='TFT', color='tab:orange')
    ax.plot([lo, hi], [lo, hi], color='black', linestyle='--', linewidth=1.3, label='Perfect (45°)')
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted (P50)')
    ax.set_title(f'Predicted vs Actual | {target_name}')
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_residual_histograms(df_check, target_name='da_price')
plot_predicted_vs_actual_scatter(df_check, target_name='da_price')


/var/folders/mh/98qk5jbj4z336rv06wpgxbz40000gn/T/ipykernel_42034/2120691853.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/mh/98qk5jbj4z336rv06wpgxbz40000gn/T/ipykernel_42034/2120691853.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Temporal Error Heatmaps (Hour × Weekday)

In [11]:
def _mae_heatmap_matrix(df: pd.DataFrame, pred_col: str) -> pd.DataFrame:
    d = df[['timestamp', 'y_true', pred_col]].dropna().copy()
    d['hour'] = d['timestamp'].dt.hour
    d['weekday'] = d['timestamp'].dt.dayofweek
    d['ae'] = (d[pred_col] - d['y_true']).abs()
    piv = d.pivot_table(index='hour', columns='weekday', values='ae', aggfunc='mean')
    piv = piv.reindex(index=range(24), columns=range(7))
    return piv


def plot_temporal_error_heatmaps(df: pd.DataFrame, target_name: str):
    xgb_hm = _mae_heatmap_matrix(df, 'xgb_p50')
    tft_hm = _mae_heatmap_matrix(df, 'tft_p50')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    sns.heatmap(xgb_hm, cmap='viridis', ax=axes[0], cbar_kws={'label': 'MAE'})
    axes[0].set_title(f'XGBoost MAE Heatmap | {target_name}')
    axes[0].set_xlabel('Weekday (Mon=0)')
    axes[0].set_ylabel('Hour of day')

    sns.heatmap(tft_hm, cmap='viridis', ax=axes[1], cbar_kws={'label': 'MAE'})
    axes[1].set_title(f'TFT MAE Heatmap | {target_name}')
    axes[1].set_xlabel('Weekday (Mon=0)')
    axes[1].set_ylabel('Hour of day')

    plt.show()


plot_temporal_error_heatmaps(df_check, target_name='da_price')


/var/folders/mh/98qk5jbj4z336rv06wpgxbz40000gn/T/ipykernel_42034/4155084343.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Specific Flaw Checks
- Quantile crossing diagnostics
- Activation-rate distribution diagnostics

In [12]:
def quantile_crossing_report(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model in ['xgb', 'tft']:
        d = df.copy()
        c10 = f'{model}_p10'
        c40 = f'{model}_p40'
        c60 = f'{model}_p60'
        c90 = f'{model}_p90'

        n = len(d)
        p10_p90 = int(((d[c10] > d[c90]) & d[c10].notna() & d[c90].notna()).sum()) if {c10, c90}.issubset(d.columns) else np.nan
        p40_p60 = int(((d[c40] > d[c60]) & d[c40].notna() & d[c60].notna()).sum()) if {c40, c60}.issubset(d.columns) else np.nan

        rows.append({
            'model': model.upper(),
            'n_rows': n,
            'crossings_p10_gt_p90': p10_p90,
            'crossings_p40_gt_p60': p40_p60,
            'pct_p10_gt_p90': (p10_p90 / n * 100.0) if isinstance(p10_p90, (int, np.integer)) and n > 0 else np.nan,
            'pct_p40_gt_p60': (p40_p60 / n * 100.0) if isinstance(p40_p60, (int, np.integer)) and n > 0 else np.nan,
        })
    return pd.DataFrame(rows)


def plot_activation_rate_distribution(df: pd.DataFrame, target_name: str):
    if 'activation_rate' not in target_name:
        print(f'Skipping activation-rate distribution for target={target_name}')
        return

    d = df[['xgb_p50', 'tft_p50']].dropna().copy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)

    sns.histplot(d['xgb_p50'], bins=50, kde=True, ax=axes[0], color='tab:blue')
    axes[0].set_title('XGBoost P50 Distribution (Activation Rate)')
    axes[0].set_xlabel('Predicted activation rate (P50)')

    sns.histplot(d['tft_p50'], bins=50, kde=True, ax=axes[1], color='tab:orange')
    axes[1].set_title('TFT P50 Distribution (Activation Rate)')
    axes[1].set_xlabel('Predicted activation rate (P50)')

    plt.tight_layout()
    plt.show()


cross_df = quantile_crossing_report(df_check)
cross_df


,model,n_rows,crossings_p10_gt_p90,crossings_p40_gt_p60,pct_p10_gt_p90,pct_p40_gt_p60
0,XGB,14522,0,0,0.0,0.0
1,TFT,14522,0,0,0.0,0.0


## Iterative Analysis Over All Targets

In [13]:
def compute_summary_stats(df: pd.DataFrame, target_name: str) -> dict[str, Any]:
    d = df[['y_true', 'xgb_p50', 'tft_p50']].dropna().copy()
    if d.empty:
        return {
            'target': target_name,
            'n': 0,
            'xgb_mae': np.nan,
            'tft_mae': np.nan,
            'xgb_rmse': np.nan,
            'tft_rmse': np.nan,
            'tft_minus_xgb_mae': np.nan,
        }

    xgb_err = d['xgb_p50'] - d['y_true']
    tft_err = d['tft_p50'] - d['y_true']

    out = {
        'target': target_name,
        'n': int(len(d)),
        'xgb_mae': float(np.mean(np.abs(xgb_err))),
        'tft_mae': float(np.mean(np.abs(tft_err))),
        'xgb_rmse': float(np.sqrt(np.mean(np.square(xgb_err)))),
        'tft_rmse': float(np.sqrt(np.mean(np.square(tft_err)))),
    }
    out['tft_minus_xgb_mae'] = out['tft_mae'] - out['xgb_mae']
    return out


def run_full_target_deepdive(targets=None, split: str = SPLIT, lead_time: int = LEAD_TIME, plot_each: bool = True) -> pd.DataFrame:
    targets = targets or TARGET_ORDER
    rows = []

    for target_name in targets:
        try:
            df = load_and_align_predictions(
                target_name=target_name,
                xgb_run_dir=XGB_RUN_DIR,
                tft_run_dir=TFT_RUN_DIR,
                split=split,
                lead_time=lead_time,
                truth_path_override=TRUTH_PATH_OVERRIDE,
            )
            rows.append(compute_summary_stats(df, target_name=target_name))

            if plot_each:
                plot_time_slice_with_bands(df, target_name=target_name)
                plot_residual_histograms(df, target_name=target_name)
                plot_predicted_vs_actual_scatter(df, target_name=target_name)
                plot_temporal_error_heatmaps(df, target_name=target_name)
                display(quantile_crossing_report(df))
                plot_activation_rate_distribution(df, target_name=target_name)
        except Exception as e:
            rows.append({'target': target_name, 'n': 0, 'error': str(e)})

    return pd.DataFrame(rows)


summary_df = run_full_target_deepdive(plot_each=False)
summary_df


,target,n,xgb_mae,tft_mae,xgb_rmse,tft_rmse,tft_minus_xgb_mae
0,da_price,14521,15.519958,15.519958,27.658706,27.658706,0.0
1,afrr_capacity_price_pos,14521,11.862445,11.862445,72.433940,72.433940,0.0
2,afrr_capacity_price_neg,14521,10.318515,10.318515,47.689420,47.689420,0.0
3,afrr_activation_price_pos,14521,390.253089,390.253089,548.148811,548.148811,0.0
4,afrr_activation_price_neg,14521,231.728511,231.728511,339.490602,339.490602,0.0
5,afrr_activation_rate_pos,14521,0.006045,0.006045,0.010848,0.010848,0.0
6,afrr_activation_rate_neg,14521,0.005735,0.005735,0.009370,0.009370,0.0


## Focused Plot Runner
Use this for one target when writing thesis figures.

In [14]:
target_name = 'da_price'  # change as needed

df_target = load_and_align_predictions(
    target_name=target_name,
    xgb_run_dir=XGB_RUN_DIR,
    tft_run_dir=TFT_RUN_DIR,
    split=SPLIT,
    lead_time=LEAD_TIME,
    truth_path_override=TRUTH_PATH_OVERRIDE,
)

plot_time_slice_with_bands(df_target, target_name=target_name)
plot_residual_histograms(df_target, target_name=target_name)
plot_predicted_vs_actual_scatter(df_target, target_name=target_name)
plot_temporal_error_heatmaps(df_target, target_name=target_name)
display(quantile_crossing_report(df_target))
plot_activation_rate_distribution(df_target, target_name=target_name)


/var/folders/mh/98qk5jbj4z336rv06wpgxbz40000gn/T/ipykernel_42034/598799236.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/mh/98qk5jbj4z336rv06wpgxbz40000gn/T/ipykernel_42034/2120691853.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/mh/98qk5jbj4z336rv06wpgxbz40000gn/T/ipykernel_42034/2120691853.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/mh/98qk5jbj4z336rv06wpgxbz40000gn/T/ipykernel_42034/4155084343.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,model,n_rows,crossings_p10_gt_p90,crossings_p40_gt_p60,pct_p10_gt_p90,pct_p40_gt_p60
0,XGB,14522,0,0,0.0,0.0
1,TFT,14522,0,0,0.0,0.0


Skipping activation-rate distribution for target=da_price
